"""
This code is provided as supplemental material to the publication "Machine learning for small data sets: an exemplary study on the classification of highly complex surface micromorphologies" 
by M. Henkel, M. Sprenger, and O. Lieleg submitted to Materials Today Advances on October 17th, 2025.

"""

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import os
import numpy as np
import logging
from tqdm import tqdm
import yaml
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

In [ ]:
class Config:
    """Configuration class to store hyperparameters"""
    def __init__(self, config_path='config_predictor.yaml'):
        #with command: automatically opens and close file 
        with open(config_path, 'r') as f:
            # .safe_load creates dictionary of f
            config = yaml.safe_load(f)
            
        
        self.batch_size = config.get('batch_size')
        self.num_classes = config.get('num_classes')
        self.image_size = config.get('image_size')
        self.kernel_size = config.get('kernel_size')
        self.dropout_rate_FC = config.get('dropout_rate_FC')
        self.dropout_rate_CONV = config.get('dropout_rate_CONV')

In [ ]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        
        padding = (Config().kernel_size - 1) // 2
        # nn.Sequential: Groups layers into a single module
        
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size= Config().kernel_size, padding=padding),    
            nn.BatchNorm2d(16),                             # Normalizes activations for better training
            nn.ReLU(),                                      # Activation function
            nn.MaxPool2d(2, 2),                             # Reduces spatial dimensions by 2
            nn.Dropout2d(Config().dropout_rate_CONV)                              # Spatial dropout for regularization
        )
        
        # Second conv block: similar structure, different channels
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size= Config().kernel_size, padding=padding),#3
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(Config().dropout_rate_CONV)
        )
        
        # Third conv block
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size= Config().kernel_size, padding=padding),#3
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(Config().dropout_rate_CONV)
        )
        
        # Global Average Pooling Layer
        self.gap = nn.AdaptiveAvgPool2d(1)  # Output: [batch, channels, 1, 1]
        
        # Fully connected layers (adjust input size to 64, the number of channels after conv3)
        self.fc1 = nn.Sequential(
            nn.Linear(64, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(Config().dropout_rate_FC)
        )
        self.fc2 = nn.Sequential(
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(Config().dropout_rate_FC)
        )
        self.fc3 = nn.Linear(128, Config().num_classes)
        
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.gap(x)             
        x = x.view(x.size(0), -1)    
        x = self.fc1(x)                 
        x = self.fc2(x)
        features = x
        x = self.fc3(x)
        
        return x, features

In [ ]:
class DatasetManager:
    @staticmethod
    def get_transforms():
            return transforms.Compose([
                transforms.CenterCrop((419, 559)),  # (height, width) (419, 559)
                transforms.Resize((Config().image_size, Config().image_size)),
                transforms.ToTensor(),
            ])

    @staticmethod
    def load_data(data_path):

        #Retrieve class names
        class_names = sorted(
        [d for d in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, d))]
        )
        print("Class names found:", class_names)
        
        # Load full dataset
        full_dataset = ImageFolderWithPath(root = data_path, allow_empty=True)
        full_dataset.transform = DatasetManager.get_transforms()
       

        # Create dataloaders
        data_loader = DataLoader(
            full_dataset,
            batch_size=Config().batch_size,
            shuffle=False,
            num_workers=0,  
            drop_last=False 
        )
        
        return data_loader, class_names

In [ ]:
class ImageFolderWithPath(torchvision.datasets.ImageFolder):
    """ImageFolder, modified to safe the file paths additionally."""
    def __getitem__(self, index):
        image, label = super().__getitem__(index)
        path, _ = self.samples[index]
        return image, label, path

In [ ]:
def get_device():
        if torch.backends.mps.is_available():
            device = torch.device("mps")
            logging.info(f'Using device: MPS (Apple Silicon GPU)')
        elif torch.cuda.is_available():
            device = torch.device("cuda")
            logging.info(f'Using device: CUDA GPU')
        else:
            device = torch.device("cpu")
            logging.info(f'Using device: CPU')
        return device

In [ ]:
def main():
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
    device = get_device()

    try:
        # Load data
        data_path = 'Test_Data'  # Replace with your actual data path

        # Classes used during training
        training_classes = ['ABR:ADH_png', 'ABR:ERO_png', 'ABR_png', 'ADH:ERO_png','ADH_png','ERO_png', 'NOD_png']

        data_loader, classes = DatasetManager.load_data(data_path)
        logging.info(f'Loaded dataset with {Config().num_classes} classes')

        if training_classes != classes :
            logging.error(f'Class mismatch between training and test datasets. {training_classes} vs {classes}')
            return
       
        # Initialize model

        model = ConvNet().to(device)
        total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"number of trainable parameters of the model: {total_trainable}")

        model_path = 'saved_models/best_model.pth'
        if os.path.exists(model_path):
            model_loader = torch.load(model_path, map_location=device)
            model.load_state_dict(model_loader ['model_state_dict'])
            logging.info(f'Model loaded from {model_path}')
        else:
            logging.error(f'Model file not found at {model_path}')
            return

        # Add test set evaluation
        logging.info('Evaluating model on test set...')
        model.eval()

        # For storing results
        all_predictions = []
        all_labels = []
        all_filenames = []

        #For storing loits for ECE
        all_logits = []
        all_label_indices = []
        
        with torch.no_grad():
            for images, labels, paths in data_loader:
                images = images.to(device)
                labels = labels.to(device)
                all_filenames.extend(paths)
                
                outputs, _ = model(images)
                _, predicted = torch.max(outputs.data, 1)

                # Convert indices to class names
                for pred, label in zip(predicted, labels):
                    all_predictions.append(classes[pred]) 
                    all_labels.append(classes[label])
        
        df_predictions = pd.DataFrame({
            'image': all_filenames,
            'true_label': all_labels,
            'predicted_label': all_predictions
            })
        df_predictions.to_csv('predictions.csv', index=False)
        logging.info("All predictions saved as predictions.csv.")
        
        # Calculate metrics using sklearn
        unique_combinations = sorted(list(set(all_predictions + all_labels)))
        report = classification_report(
            all_labels,
            all_predictions,
            labels=unique_combinations,
            zero_division=0,
            output_dict=True
        )
        
        # Print metrics table
        logging.info('\nMetrics Summary:')
        logging.info('\n{:<15} {:<12} {:<12} {:<12} {:<12}'.format(
            'Class', 'Precision', 'Recall', 'F1-Score', 'Support'))
        logging.info('-' * 63)

        # Print per-class metrics
        for combo in unique_combinations:
            metrics = report[combo]
            logging.info('{:<15} {:<12.2f} {:<12.2f} {:<12.2f} {:<12.0f}'.format(
                combo,
                metrics['precision'],
                metrics['recall'],
                metrics['f1-score'],
                metrics['support']
            ))
        
        # Print average metrics
        logging.info('-' * 63)
        logging.info('{:<15} {:<12.2f} {:<12.2f} {:<12.2f} {:<12.0f}'.format(
            'Macro Avg.',
            report['macro avg']['precision'],
            report['macro avg']['recall'],
            report['macro avg']['f1-score'],
            report['macro avg']['support']
        ))
        logging.info('{:<15} {:<12.2f} {:<12.2f} {:<12.2f} {:<12.0f}'.format(
            'Weighted Avg.',
            report['weighted avg']['precision'],
            report['weighted avg']['recall'],
            report['weighted avg']['f1-score'],
            report['weighted avg']['support']
        ))
        
        # Create confusion matrix
        cm = confusion_matrix(all_labels, all_predictions, labels=unique_combinations)
        
        # Plot confusion matrix
        plt.figure(figsize=(12, 10))
        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=unique_combinations
        )
        disp.plot(xticks_rotation=90)
        plt.title('Confusion Matrix')
        plt.tight_layout()
        save_dir = 'visualizations'
        os.makedirs(save_dir, exist_ok=True)
        plt.savefig('visualizations/confusion_matrix.png')
        plt.close()

        

    except Exception as e:
        logging.error(f'An error occurred: {str(e)}')
        raise
    finally:
        torch.cuda.empty_cache()
 
if __name__ == '__main__':
    main()